# 🚀 Обучение крипто-бота на бесплатном GPU (Google Colab)

Этот ноутбук настроен для обучения модели на 5 топовых криптовалютах и 7 таймфреймах.

**Важно:** Перед запуском убедитесь, что включен GPU:
1. Меню `Runtime` (Среда выполнения) -> `Change runtime type` (Тип среды выполнения)
2. Выберите `GPU` в поле `Hardware accelerator`
3. Нажмите `Save`

## 1. Установка зависимостей

In [ ]:
!pip install ccxt pandas numpy torch scikit-learn ta-lib-binary --quiet
# Примечание: ta-lib может потребовать ручной установки в некоторых средах, используем упрощенную версию или альтернативы

## 2. Подключение Google Drive (для сохранения моделей)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Создадим папку для сохранения результатов
import os
os.makedirs('/content/drive/MyDrive/crypto_bot_models', exist_ok=True)
print("✅ Google Drive подключен. Модели будут сохранены в папку crypto_bot_models")

## 3. Клонирование репозитория или загрузка кода

*Если ваш код уже на GitHub, раскомментируйте строку ниже и вставьте ссылку.*
*Или мы можем создать необходимые файлы прямо здесь.*

In [ ]:
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
# %cd YOUR_REPO

# Если репозитория нет, создадим структуру файлов прямо в ноутбуке для демонстрации:
import os
os.makedirs('src/data', exist_ok=True)
os.makedirs('src/trading', exist_ok=True)
os.makedirs('configs', exist_ok=True)
os.makedirs('models', exist_ok=True)
print("✅ Структура папок создана")

## 4. Создание конфигурационного файла

In [ ]:
config_content = """
symbols:
  - BTC/USDT
  - ETH/USDT
  - BNB/USDT
  - SOL/USDT
  - XRP/USDT

timeframes:
  - 5m
  - 15m
  - 1h
  - 4h
  - 12h
  - 1d
  - 1w

model:
  type: "lstm_attention"
  hidden_size: 128
  num_layers: 2
  dropout: 0.2
  lookback_periods:
    5m: 200
    15m: 150
    1h: 100
    4h: 80
    12h: 60
    1d: 50
    1w: 30

training:
  epochs: 50
  batch_size: 32
  learning_rate: 0.001
  test_split: 0.2
  early_stopping_patience: 5
"""

with open('configs/config.yaml', 'w') as f:
    f.write(config_content)
print("✅ config.yaml создан")

## 5. Реализация логики сбора данных и обучения (Упрощенная версия для ноутбука)

Здесь мы объединим логику сбора данных, подготовки признаков и обучения в одном блоке для удобства запуска в Colab.

In [ ]:
import ccxt
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime
import time

# --- 1. Сбор данных ---
def fetch_data(symbol, timeframe, limit=1000):
    exchange = ccxt.binance()
    print(f"📥 Загрузка {symbol} ({timeframe})...")
    try:
        bars = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)
        df = pd.DataFrame(bars, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
        return df
    except Exception as e:
        print(f"Ошибка при загрузке {symbol}: {e}")
        return None

# --- 2. Инжиниринг признаков ---
def add_features(df):
    df = df.copy()
    # RSI
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))
    
    # SMA
    df['sma_20'] = df['close'].rolling(window=20).mean()
    df['sma_50'] = df['close'].rolling(window=50).mean()
    
    # Volatility
    df['volatility'] = df['close'].pct_change().rolling(window=14).std()
    
    # Target: Следующее изменение цены (1 = рост, 0 = падение)
    df['target'] = (df['close'].shift(-1) > df['close']).astype(int)
    
    df.dropna(inplace=True)
    return df

# --- 3. Модель (Multi-Timeframe LSTM) ---
class CryptoLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super(CryptoLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = self.fc(lstm_out[:, -1, :]) # Берем последний шаг времени
        return self.sigmoid(out)

# --- 4. Процесс обучения ---
def train_model():
    symbols = ['BTC/USDT', 'ETH/USDT', 'BNB/USDT', 'SOL/USDT', 'XRP/USDT']
    timeframes = ['5m', '15m', '1h', '4h', '1d'] # Для демо берем 5 основных, чтобы не ждать вечно
    
    all_data = {}
    
    # Сбор и подготовка данных
    for symbol in symbols:
        all_data[symbol] = {}
        for tf in timeframes:
            df = fetch_data(symbol, tf)
            if df is not None:
                df = add_features(df)
                all_data[symbol][tf] = df
    
    print("\n🚀 Начало обучения...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Используемое устройство: {device}")
    
    results = {}
    
    # Обучаем отдельную модель для каждой монеты (упрощенный подход для старта)
    # В продакшене можно сделать одну большую модель с эмбеддингами монет
    for symbol in symbols:
        print(f"\n🧠 Обучение модели для {symbol}...")
        
        # Объединяем данные всех таймфреймов для этой монеты (простая конкатенация по времени для примера)
        # В реальной задаче нужно аккуратно ресемплировать или использовать отдельные входы
        # Здесь для простоты возьмем данные 1h как основные и добавим признаки с других, если они совпадают по времени
        # Или просто обучим на 1h для демонстрации работы пайплайна
        base_tf = '1h'
        if base_tf not in all_data[symbol]:
            continue
            
        df_main = all_data[symbol][base_tf]
        
        features = ['open', 'high', 'low', 'close', 'volume', 'rsi', 'sma_20', 'sma_50', 'volatility']
        
        scaler = MinMaxScaler()
        scaled_data = scaler.fit_transform(df_main[features])
        
        # Создание последовательностей (Lookback = 60)
        lookback = 60
        X, y = [], []
        for i in range(lookback, len(scaled_data) - 1):
            X.append(scaled_data[i-lookback:i])
            y.append(df_main['target'].iloc[i])
            
        X = np.array(X)
        y = np.array(y)
        
        # Split
        split = int(len(X) * 0.8)
        X_train, X_test = X[:split], X[split:]
        y_train, y_test = y[:split], y[split:]
        
        # Tensor Dataset
        train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train).unsqueeze(1))
        test_dataset = TensorDataset(torch.FloatTensor(X_test), torch.FloatTensor(y_test).unsqueeze(1))
        
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=32)
        
        # Model
        model = CryptoLSTM(input_size=len(features), hidden_size=64, num_layers=2, dropout=0.2).to(device)
        criterion = nn.BCELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        
        epochs = 20 # Уменьшено для демо
        best_loss = float('inf')
        
        for epoch in range(epochs):
            model.train()
            total_loss = 0
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                
                optimizer.zero_grad()
                output = model(batch_X)
                loss = criterion(output, batch_y)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            
            avg_loss = total_loss / len(train_loader)
            if (epoch + 1) % 5 == 0:
                print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
                
            if avg_loss < best_loss:
                best_loss = avg_loss
                torch.save(model.state_dict(), f"/content/drive/MyDrive/crypto_bot_models/{symbol.replace('/', '_')}_model.pth")
                
        results[symbol] = best_loss
        print(f"✅ Модель для {symbol} сохранена!")

    print("\n🎉 Обучение завершено!")
    print("Результаты (лучший Loss):")
    for k, v in results.items():
        print(f"{k}: {v:.4f}")

# Запуск
if __name__ == "__main__":
    train_model()